# Assemble datasets from simulations
Combine data from simulations of different network architectures

In [1]:
import numpy as np
import pandas as pd
import os
import pickle
from tqdm import tqdm
from joblib import Parallel, delayed
import re

from stoch_sim_model import *

In [2]:
# Set parameters
sim_kind = 'agent'
reg_model = ''
runs = '-1-'
comment = "acute_all-sparse-reg"

d = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/'

sim_sum_list = []
parameters_nets = []
prim_diff_bias_list = []
#sec_diff_bias_list = []
cell_series_list = []
# lineage_diff_nets = []

In [3]:
# Figure out which jobs didn't run:
d_rerun = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/raw/'
run_list = [int(re.search('sim_batch_(.*?)\.', f).group(1)) for f in os.listdir(d_rerun) if 'sim_batch' in f and comment in f and runs in f]
out = [str(x) for x in [k for k in np.arange(0, 284)] if x not in run_list]
print(len(out))
print(' '.join((out)))

0



In [4]:
num_cpu = 150
file_list = [f for f in os.listdir(os.path.join(d, "raw")) if runs in f and comment in f and 'sim_batch' in f]
num_files = len(file_list)

def import_dict_func(f,d):
    
    file_path = os.path.join(os.path.join(d, "raw"), f)
    with open(file_path, 'rb') as filename:  
        import_dict = pickle.load(filename)

    parameters = np.array(import_dict["parameters"])
    sim_sum = np.array(import_dict["summary_stats"])

    out = np.hstack((parameters, sim_sum))

    return out

# create dataframe of infection response statistics
var_names = np.concatenate((param_names_for_df, stat_names_for_df))
# mean_df = pd.DataFrame(np.vstack(Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d) 
#                                                                                                         for file_name in file_list)), 
#                        columns = [i for i in var_names]).groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).mean()
full_df = pd.DataFrame(np.vstack(Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d) 
                                                                                                        for file_name in file_list)), 
                       columns = [i for i in var_names])

# # Save datasets
full_df.to_pickle(os.path.join(d, "raw", "stacked_full_data"+runs+"runs"+'-'+comment)+'.pkl')

In [5]:
with pd.option_context('display.max_columns', None):
    display(full_df)

,S_0,I_0,b_I,d_S,d_I,d_IE,K_I,b_H,d_H,K_H,N_0,max_Na,b_myc,d_myc,myc_thresh,t_bind,t_unbind,t_Na_div,t_E_div,t_M_div,t_E_die,t_cycle,psi_myc_I,psi_myc_HI,psi_myc_HE,L0_Na,psi_NE_I,psi_NE_HI,psi_NE_HE,L0_NE,psi_EM_I,psi_EM_HI,psi_EM_HE,L0_EM,psi_Edie_I,psi_Edie_HI,psi_Edie_HE,L0_Edie,p_load,T_max_pI,T_min_pI,harm_pI,harm_pS,max_pE,T_pE_max,T_pE_start,max_eM,T_pEcyteM,T_pE_end,frac_cM,int_pHE,int_pHI,min_pS
0,10000000.0,1000.0,1.500000e-07,0.01,0.15,12.0,10000.0,1.0,2.0,100000.0,100.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.0,0.0,0.0,4.0,-3.0,1.5,0.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,-4.0,6.815947e+06,8.73,0.0,1.170315e+07,1169.454063,796.0,2.24,0.47,768.0,1.087513,0.00,0.533666,458.502012,500819.333111,1.505112e+05
1,10000000.0,1000.0,1.500000e-07,0.01,0.15,12.0,10000.0,1.0,2.0,100000.0,100.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.0,0.0,0.0,4.0,-3.0,1.5,0.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,-2.0,6.817133e+06,8.72,0.0,1.170440e+07,704.233287,535.0,1.71,0.53,442.0,0.610498,0.00,0.591022,297.507126,500904.659504,1.504821e+05
2,10000000.0,1000.0,1.500000e-07,0.01,0.15,12.0,10000.0,1.0,2.0,100000.0,100.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.0,0.0,0.0,4.0,-3.0,1.5,0.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,6.817941e+06,8.71,0.0,1.170486e+07,404.852333,389.0,1.20,0.30,266.0,0.294361,0.00,0.456359,167.871782,500961.972696,1.504624e+05
3,10000000.0,1000.0,1.500000e-07,0.01,0.15,12.0,10000.0,1.0,2.0,100000.0,100.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.0,0.0,0.0,4.0,-3.0,1.5,0.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,2.0,6.818170e+06,8.70,0.0,1.170576e+07,265.433740,310.0,1.39,0.26,153.0,0.210980,0.00,0.474074,132.359227,500978.502827,1.504569e+05
4,10000000.0,1000.0,1.500000e-07,0.01,0.15,12.0,10000.0,1.0,2.0,100000.0,100.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.0,0.0,0.0,4.0,-3.0,1.5,0.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,6.818094e+06,8.70,0.0,1.170559e+07,301.436868,347.0,1.03,0.27,192.0,0.215573,0.00,0.364764,119.611100,500972.830634,1.504587e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73124995,10000000.0,500000.0,1.000000e-01,0.01,0.01,12.0,100000000.0,1.0,2.0,100000.0,1000.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.5,-1.5,-1.5,0.0,0.0,0.0,0.0,4.0,5.841625e+06,30.00,0.0,1.118347e+07,7436.435639,5535.0,2.25,0.20,1879.0,0.398441,0.00,0.175508,839.658250,151250.473792,5.191485e+06
73124996,10000000.0,500000.0,1.000000e-01,0.01,0.01,12.0,100000000.0,1.0,2.0,100000.0,1000.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.5,-1.5,-1.5,2.0,0.0,0.0,0.0,-4.0,5.797410e+06,30.00,0.0,1.110062e+07,67465.039753,37950.0,3.14,0.27,37768.0,6.651359,4.87,0.169109,3358.532355,149763.321880,5.179189e+06
73124997,10000000.0,500000.0,1.000000e-01,0.01,0.01,12.0,100000000.0,1.0,2.0,100000.0,1000.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.5,-1.5,-1.5,2.0,0.0,0.0,0.0,-2.0,5.827953e+06,30.00,0.0,1.115667e+07,22430.443053,12903.0,2.69,0.23,12439.0,1.082261,0.00,0.157228,2040.434119,150899.390232,5.191931e+06
73124998,10000000.0,500000.0,1.000000e-01,0.01,0.01,12.0,100000000.0,1.0,2.0,100000.0,1000.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.5,-1.5,-1.5,2.0,0.0,0.0,0.0,0.0,5.840586e+06,30.00,0.0,1.118143e+07,8560.613140,6146.0,1.74,0.27,4671.0,0.472389,0.00,0.181480,880.632089,151224.285761,5.191535e+06


In [6]:
# Create additional variables
virs = np.unique(full_df[['I_0','d_I','K_I','b_I','K_H','N_0']].values, axis = 0)

full_df['antigenicity_over_harm'] = antigenicity_over_harm(full_df)
full_df['T_pE_clear'] = full_df['T_max_pI'] - full_df['T_pE_start']
full_df['max_eM_fold'] = np.log10(1 + full_df['max_eM']/full_df['N_0'])
full_df['stim_pI'] = np.log(1 + (full_df['p_load']/full_df['K_I']))
full_df['stim_pHI'] = np.log(1 + (full_df['int_pHI']/full_df['K_H']))
full_df['stim_pHE'] = np.log(1 + (full_df['int_pHE']/full_df['K_H']))
full_df['max_pE_fold'] = np.log10(1 + full_df['max_pE']/full_df['N_0'])
full_df['scaled_min_pS'] = full_df['min_pS']/full_df['S_0']
full_df['log_T_pEcyteM'] = np.log10(1 + full_df['T_pEcyteM']/sim_duration)

# identify Biologically evidenced networks
keep_vars = ['harm_pI', 'harm_pS', 'max_eM_fold', 'frac_cM', 'log_T_pEcyteM',
             'T_min_pI', 'T_max_pI', 'T_pE_start', 'T_pE_clear',
             'stim_pI', 'stim_pHI', 'stim_pHE',
             'scaled_min_pS', 'antigenicity_over_harm']

In [7]:
# save data sets
full_infection_scenarios = []
mean_of_infection_scenarios = []
std_of_infection_scenarios = []
no_eff_data = [[] for i in np.arange(len(virs))]
b_S = d_S*S_0

for l, (I_0, d_I, K_I, b_I, K_H, N_0) in enumerate(tqdm(virs)):
    data = full_df.loc[(full_df["d_I"] == d_I)*(full_df["K_I"] == K_I)*(full_df["b_I"] == b_I)*(full_df["K_H"] == K_H)*(full_df["N_0"] == N_0)*(full_df["I_0"] == I_0), 
    ['b_I','d_I', 'K_I', 'I_0','S_0', 'N_0', 'd_S', 'K_H'] + Na_reg + NE_reg + EM_reg + EE_reg + keep_vars]

    # compute infection harm without T cell response
    no_eff_data[l] = lin_stoch_sim(N_0 = 0, I_0 = I_0, K_I = K_I, d_I = d_I, b_I = b_I,
                                   infection_model = "cancer" if b_I >= b_C else "acute")
    no_eff_stats = no_eff_data[l]["summary_stats"]

    data.loc[:,"harm_pI_noprotection"] = no_eff_stats[3]/S_0
    data.loc[:,"peff_clearance"] = (no_eff_stats[3] - data['harm_pI'].to_numpy())/S_0
    data.loc[:,"peff_toxicity"] = data['harm_pS'].to_numpy()/S_0

    mean_of_infection_scenarios.append(data.groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).mean())
    std_of_infection_scenarios.append(data.groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).std())
    full_infection_scenarios.append(data)

# stack datasets
pd.concat(mean_of_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/processed_data'+runs+'runs'+'-'+comment+'.pkl')
pd.concat(std_of_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/std/processed_data'+runs+'runs'+'-'+comment+'.pkl')
pd.concat(full_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/processed_full_data'+runs+'runs'+'-'+comment+'.pkl')

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(mean_of_infection_scenarios, f)

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/std/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(std_of_infection_scenarios, f)

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/list_processed_full_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(full_infection_scenarios, f)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 234/234 [29:29<00:00,  7.56s/it]


In [8]:
# Clear memory
del full_df, mean_of_infection_scenarios, std_of_infection_scenarios, full_infection_scenarios